> 在线讲义：[第 5 章 · GPU 内存层级：寄存器、shared memory、L1 / L2、显存](https://wikieden.github.io/cuda-to-tensor-core/lessons/05-memory-hierarchy.html) · 本书《CUDA 从线程到 Tensor Core》

# 第 5 章 · GPU 内存层级：寄存器、shared memory、L1 / L2、显存

本章回答：kernel 里的变量和数组到底存在哪？为什么"离计算单元越近越快、越小"？一个 H100 上每一层有多大、多快？写 kernel 时哪些东西是自己管、哪些是硬件替你管？

配套讲义：仓库里 `cuda-programming/notes/lessons/05-memory-hierarchy.md`（讲解、动画、练习都在讲义里）。对应原文：PG §1.2.3.3 On-Chip Memory（`materials/text/cuda-programming-guide.txt` 第 1314–1349 行）、§2.3.3 GPU Device Memory Spaces（第 3209–3742 行，含 2.3.3.1–2.3.3.8）；Hopper tuning guide §4.1.1、§4.2（`text/hopper-tuning-guide.txt` 第 141–155、235–250 行）；Gordić《Inside NVIDIA GPUs》Memory 一节（`text/blog-gordic-inside-nvidia-gpus-matmul.txt` 第 37–58 行）；Oxford lec2（`text/oxford-lec2.txt`）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU：代码用 NumPy 在 CPU 上模拟 GPU 的规则（线程编号、32 字节一箱、32 个 bank……）。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 寄存器 | 64K × 4 B | `H100["寄存器（每 SM）"]` | 256 KB，和片上 shared + L1 一样大 |
| 第二步 shared 合计 | 132 × 228 KB | `sm_totals()` | 29.4 MB（讲义"约 30 MB"），L2 50 MB |
| 第四步 读一次用多次 | 8 个线程用同一块 8 个数 | `global_reads(8, False)` / `global_reads(8, True)` | 64 次 vs 8 次 |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**常量 `KB`, `MB`, `GB`**（来自 `ch05_memory_hierarchy.py`）

In [ ]:
KB, MB, GB = 2**10, 2**20, 2**30

**常量 `H100`**（来自 `ch05_memory_hierarchy.py`）

In [ ]:
H100 = {
    "寄存器（每 SM）": 65536 * 4,          # 64K 个 32 位寄存器
    "shared memory（每 SM 最多）": 228 * KB,
    "L2（全 GPU）": 50 * MB,
    "HBM3（全 GPU）": 80 * GB,
}

**常量 `MAX_REGS_PER_THREAD`**（来自 `ch05_memory_hierarchy.py`）

In [ ]:
MAX_REGS_PER_THREAD = 255

**函数 `demo_reuse`**（来自 `ch05_memory_hierarchy.py`）：动画最后一帧：一块数据搬上 shared 后，被 8 个线程各用一遍（这里每个线程求一个加权和）。

In [ ]:
def demo_reuse(n_threads: int = 8) -> np.ndarray:
    """动画最后一帧：一块数据搬上 shared 后，被 8 个线程各用一遍（这里每个线程求一个加权和）。"""
    g = np.arange(8, dtype=np.float32)            # 显存里的数据
    s = g.copy()                                  # 读一次，放进"shared"
    w = np.arange(1, n_threads + 1)[:, None]      # 每个线程自己的权重（放寄存器）
    return (s[None, :] * w).sum(axis=1)

**函数 `global_reads`**（来自 `ch05_memory_hierarchy.py`）：n_threads 个线程都要用同一块 8 个数的数据。

In [ ]:
def global_reads(n_threads: int, reuse_via_shared: bool) -> int:
    """n_threads 个线程都要用同一块 8 个数的数据。

    不用 shared：每个线程各自从显存读 8 个数；用 shared：block 合作读一次（8 次），大家从 shared 里拿。
    """
    tile = 8
    return tile if reuse_via_shared else n_threads * tile

**常量 `SMS`**（来自 `ch05_memory_hierarchy.py`）

In [ ]:
SMS = 132

**函数 `sm_totals`**（来自 `ch05_memory_hierarchy.py`）：几个讲义里的"反直觉"数字。

In [ ]:
def sm_totals() -> dict[str, float]:
    """几个讲义里的"反直觉"数字。"""
    return {
        "每 SM 寄存器 KB": H100["寄存器（每 SM）"] / KB,
        "132 个 SM 的 shared 合计 MB": SMS * H100["shared memory（每 SM 最多）"] / MB,
        "L2 MB": H100["L2（全 GPU）"] / MB,
    }

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步 H100 每一层的容量")
for k, v in H100.items():
    unit, size = next((u, s) for u, s in (("GB", GB), ("MB", MB), ("KB", KB)) if v >= s)
    print(f"  {k:<22} {v / size:>5.0f} {unit}")
for k, v in sm_totals().items():
    print(f"  {k:<22} {v:,.1f}")
print(f"  每线程最多 {MAX_REGS_PER_THREAD} 个寄存器")
print("  寄存器总量 = shared + L1 的片上总量（256 KB）：", H100["寄存器（每 SM）"] == 256 * KB)

In [ ]:
print("\n第四步 读一次、用很多次：8 个线程都要同一块 8 个数")
print("  各读各的：显存读", global_reads(8, False), "次；经过 shared：", global_reads(8, True), "次")
print("  8 个线程各自算出的结果：", demo_reuse())

## 3. 代码里才看得到的

1. **延迟没有写进代码**：讲义里延迟只给了量级（〔旧知〕），本地文档里没有官方表，所以代码只用容量做计算。
2. **`KB = 2**10`**：容量按 1024 进位算；带宽（GB/s、TB/s）习惯上按 1000 进位，第 4、9 章的带宽账用的是 `1e9`。

## 4. 自己改一改

- 把 `global_reads` 的线程数改成 32、256，看"各读各的"和"经过 shared"差几倍。
- 按 B200 的数（148 个 SM、shared 也是 228 KB）重算 shared 合计。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 手里、台面、小冰箱、大冷库、城外仓库：越往外越大越慢；
> 寄存器编译器管，shared 自己管，L1 / L2 硬件管，显存怎么读你来管；
> local 不在本地，在显存里，数组下标别用变量；
> 优化一句话：读一次，用很多次。